# Удалённая работа: какие сегменты брать и где нужен адресный добор

**Вопрос.** Какие удалённые сегменты можно передать в ML, какие признаки извлекать из описаний и где нужен адресный добор.

**Что делаем:**
1. Выделяем удалённую когорту и проверяем заполненность географии.
2. Сравниваем городской разрез (`роль × город`) и общероссийскую группу (`роль × удалёнка`).
3. Размечаем удалённые роли по названию вакансии: в удалённых выгрузках поле `profession` пустое.
4. Для каждой роли считаем объём, работодателей и форматы оплаты, обосновываем подсегменты.
5. Оцениваем, как часто в описаниях встречаются кандидаты в текстовые признаки.
6. Собираем таблицу кандидатов, фильтры для добора и рекомендацию для дерева сегментов.

Пороги пригодности взяты из командного исследования зарплат (`salary_segment_results/README.md`):
класс A — не меньше 30 сопоставимых зарплат, не меньше 10 работодателей, крупнейший работодатель не больше 30%. Цель добора — 50.

Извлечение признаков здесь не реализуется. Оно запланировано с 23.11 по подходу Константина;
здесь только частота упоминаний простыми регулярными выражениями, чтобы выбрать кандидатов.

In [ ]:
import hashlib
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 70)


def find_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / "Data" / "dataset").exists():
            return p
    raise FileNotFoundError("Запустите из репозитория salary-navigator")


ROOT = find_root()
DATA_PATH = ROOT / "Data" / "dataset" / "vacancies.csv"   # python3 utils/build_dataset.py --materialize
OUT = ROOT / "Analytics" / "remote_segment_results"
OUT.mkdir(exist_ok=True)

N_MIN, N_TARGET, MIN_EMPLOYERS, TOP1_MAX = 30, 50, 10, 0.30

raw = pd.read_csv(DATA_PATH, dtype={"id": "string"}, low_memory=False)
DATA_SHA = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print(f"{DATA_PATH.relative_to(ROOT)}: {raw.shape}, SHA-256 {DATA_SHA}")

Data/dataset/vacancies.csv: (32087, 131), SHA-256 b607c7f0df325bb24537a92cef7c995716878ab314023e0c7548c30367f7f5dc


предобработка

In [ ]:
HOMO = str.maketrans("aeopcxykmthbABEKMHOPCTXY", "аеорсхукмтнвАВЕКМНОРСТХУ")
WORD = re.compile(r"[A-Za-zА-Яа-яЁё]+")


def fix_text(text):
    if not isinstance(text, str):
        return ""
    def fix(m):
        w = m.group(0)
        return w.translate(HOMO) if re.search(r"[А-Яа-яЁё]", w) else w
    return WORD.sub(fix, text).lower().replace("ё", "е")


df = raw.drop_duplicates(["source", "id"], keep="last").copy()
df["title_n"] = df["title"].map(fix_text)
df["desc_n"] = df["description"].map(fix_text)
df["text_n"] = df["title_n"] + " \n " + df["desc_n"]
share_homo = df["description"].fillna("").str.contains(r"[а-яё][a-z]|[a-z][а-яё]", case=False).mean()
print(f"Описаний с латинскими двойниками: {share_homo:.0%}")

Описаний с латинскими двойниками: 91%


## 1. Удалённая когорта
Удалёнкой считаем вакансию, у которой в поле Авито «Формат работы» отмечено «Удалённо»:
* `только удалённо` — выбран один вариант;
* `удалённо + офис/гибрид` — работодатель допускает и удалёнку, и другие форматы.


In [ ]:
fmt = df["Формат работы"].fillna("")
df["remote_format"] = np.select(
    [fmt.eq("Удалённо"), fmt.str.contains("Удалённо")],
    ["только удалённо", "удалённо + офис/гибрид"], default="не удалённо",
)
remote_run = df["run_id"].str.contains("remote", case=False, na=False)
cohort_tbl = pd.crosstab(df["remote_format"], remote_run.rename("сбор с remote в run_id"), margins=True)
print(cohort_tbl)

r = df[(df["remote_format"] != "не удалённо") & ~df["military"].astype(bool)].copy()
print(f"\nУдалённая когорта без military: {len(r)} вакансий "
      f"(исключено military: {int(((df['remote_format'] != 'не удалённо') & df['military'].astype(bool)).sum())}, "
      f"без отметки «Удалённо» из удалённых сборов: {int((remote_run & (df['remote_format'] == 'не удалённо')).sum())})")

сбор с remote в run_id  False  True    All
remote_format                             
не удалённо             27096   854  27950
только удалённо            29  3295   3324
удалённо + офис/гибрид      1   812    813
All                     27126  4961  32087

Удалённая когорта без military: 4105 вакансий (исключено military: 32, без отметки «Удалённо» из удалённых сборов: 854)


## 2. География: заполненность и смысл города
Для удалёнки `city` — это город, где работодатель **разместил** объявление

In [ ]:
geo_fill = pd.DataFrame({
    "заполнено": {
        "city (город с карточки)": r["city"].notna().mean(),
        "region": r["region"].notna().mean(),
        "city_resolved (с восстановлением из адреса)": r["city_resolved"].notna().mean(),
        "profession (фильтр сбора)": r["profession"].notna().mean(),
        "profession_resolved (6 массовых профессий)": r["profession_resolved"].notna().mean(),
    }
}).round(3)
print(geo_fill)
print("\nГеография поиска (фильтр сбора):")
print(r["search_geography"].value_counts(dropna=False).head(10))

city_tbl = r["city_resolved"].fillna("нет города").value_counts().to_frame("n")
city_tbl["share"] = (city_tbl["n"] / len(r)).round(3)
city_tbl["cum_share"] = city_tbl["share"].cumsum().round(3)
print(f"\nУникальных городов размещения: {r['city_resolved'].nunique()}")
print(city_tbl.head(12))
city_tbl.reset_index(names="city_resolved").to_csv(OUT / "remote_city_counts.csv", index=False)

                                             заполнено
city (город с карточки)                          0.458
region                                           0.660
city_resolved (с восстановлением из адреса)      0.966
profession (фильтр сбора)                        0.000
profession_resolved (6 массовых профессий)       0.007

География поиска (фильтр сбора):
search_geography
По всей России      2717
Рязань               723
Москва               281
Рязань, район        244
Краснодар, район      87
Краснодар             31
Камчатский край       10
Амга                   7
Санкт-Петербург        4
Якутск                 1
Name: count, dtype: int64

Уникальных городов размещения: 511
                    n  share  cum_share
city_resolved                          
Москва           1157  0.282      0.282
Санкт-Петербург   212  0.052      0.334
Краснодар         193  0.047      0.381
нет города        140  0.034      0.415
Ростов-на-Дону     77  0.019      0.434
Новосибирск        76  0.01

In [ ]:
# Одна и та же вакансия (работодатель + название) размещается в нескольких городах
r["title_key"] = r["title_n"].str.replace(r"[^а-яa-z0-9]", "", regex=True)
multi_city = (r.groupby(["employer_name_observed", "title_key"])["city_resolved"]
              .nunique().rename("n_cities").reset_index())
in_multi = r.merge(multi_city, on=["employer_name_observed", "title_key"])["n_cities"] >= 2
print(f"Вакансии, чьи копии (тот же работодатель и название) размещены в 2+ городах: {in_multi.mean():.0%}")

Вакансии, чьи копии (тот же работодатель и название) размещены в 2+ городах: 26%


**Вывод.** Город почти всегда можно восстановить (97%), но для удалёнки это город размещения.
Половина объявлений — Москва и Санкт-Петербург, остальное  по сотням городов.

## 3. Удалённые роли по названию вакансии
у 95% удалённых вакансий `profession_resolved` пустая.
Поэтому вводим отдельную разметку удалённых ролей

In [ ]:
ROLE_RULES = [
    ("Взыскание задолженности", r"взыскан|коллектор|просроч"),
    ("Оператор колл-центра / поддержки", r"колл|call|кол-центр|контакт\w*[ -]центр|оператор|поддержк|лини[июя]\b|\bчат"),
    ("Рекрутер / HR", r"рекрут|подбор|\bhr\b|эйчар|кадр"),
    ("Репетитор / преподаватель", r"репетитор|преподава|учител|педагог|наставник|куратор|тренер|логопед"),
    ("Маркетплейсы / контент", r"маркетплейс|ozon|озон|wildberries|\bwb\b|\bвб\b|авитолог|карточ|контент"),
    ("Агент: недвижимость / страхование / банк", r"\bагент(?!ств)|турагент|риелт|риэлт|недвижим|страхов|брокер|карт\w* банк|банковск\w* продукт|представитель банк"),
    ("Менеджер по продажам / работе с клиентами", r"продаж|продаве|привлечен|развити\w* бизнес|сделок|торгов\w* представ|b2b|работе с клиент|клиентск|аккаунт|менеджер по работе"),
    ("Маркетинг / SMM / реклама", r"маркетолог|маркетинг|smm|смм|таргет|реклам|продвиж|блогер|трафик|pr\b|seo"),
    ("Дизайнер", r"дизайн|иллюстрат|3d|визуализ|монтаж\w* видео|видеомонтаж|моушн"),
    ("Ассистент / администратор", r"ассистент|помощник|секретар|администратор|офис[ -]менеджер|модератор|ввод\w* данн|оператор пк"),
    ("Бухгалтер / финансы", r"бухгалт|экономист|финанс|кассир-операционист"),
    ("Юрист", r"юрист|юрисконсульт|правов|арбитраж|адвокат"),
    ("Копирайтер / редактор / переводчик", r"копирайт|редактор|корректор|автор\b|переводчик|рерайт"),
    ("Логист / диспетчер", r"логист|диспетчер|закуп"),
    ("IT / инженер / проектировщик", r"разработ|программ|developer|python|java|тестиров|\bqa\b|аналитик|инженер|проектировщ|конструктор|сметчик|\b1с\b|1c|сисадмин|devops|frontend|backend"),
    ("Надомная ручная работа", r"надом|швея|сборщ|упаков|вязал|рукодел"),
]


def role_of(title):
    for name, pat in ROLE_RULES:
        if re.search(pat, title):
            return name
    return "Прочее"


r["remote_role"] = r["title_n"].map(role_of)
role_counts = r["remote_role"].value_counts().to_frame("n")
role_counts["share"] = (role_counts["n"] / len(r)).round(3)
print(role_counts)
print("\nПримеры «Прочее»:", r.loc[r["remote_role"] == "Прочее", "title"].sample(15, random_state=1).tolist())

                                             n  share
remote_role                                          
Менеджер по продажам / работе с клиентами  899  0.219
Прочее                                     623  0.152
Оператор колл-центра / поддержки           483  0.118
Взыскание задолженности                    358  0.087
Ассистент / администратор                  239  0.058
Маркетплейсы / контент                     233  0.057
Репетитор / преподаватель                  219  0.053
Маркетинг / SMM / реклама                  216  0.053
Агент: недвижимость / страхование / банк   212  0.052
Рекрутер / HR                              202  0.049
IT / инженер / проектировщик               114  0.028
Логист / диспетчер                          89  0.022
Дизайнер                                    70  0.017
Бухгалтер / финансы                         63  0.015
Надомная ручная работа                      57  0.014
Копирайтер / редактор / переводчик          18  0.004
Юрист                       

## 4. Работодатели: концентрация, частные лица, массовые размещения
На удалёнке есть работодатели, которые размещают десятки почти одинаковых объявлений
(T-Банк — «Оператор раннего взыскания»)


In [ ]:
FIRST_NAMES = set("""александр дмитрий сергей андрей алексей максим иван евгений михаил владимир николай артем
павел роман денис игорь олег виктор антон илья кирилл никита вадим руслан юрий константин станислав
елена ольга анна наталья татьяна екатерина мария ирина светлана юлия анастасия марина алина виктория оксана
галина людмила дарья ксения надежда вера валентина полина софия кристина ангелина маша""".split())
BRAND_MERGE = {"edler ai": "edler", "партнеры voxys": "voxys", "т-банк": "т-банк", "t-банк": "т-банк"}


def employer_key(name):
    if not isinstance(name, str):
        return "нет данных"
    low = re.sub(r"[«»\"'“”]", "", name.lower().replace("ё", "е")).strip()
    low = re.sub(r"^(ооо|ао|пао|ип|зао)\s+", "", low)
    tokens = low.split()
    if low == "пользователь" or (tokens and tokens[0] in FIRST_NAMES and len(tokens) <= 2):
        return "частное лицо"
    return BRAND_MERGE.get(low, low)


r["employer_key"] = r["employer_name_observed"].map(employer_key)
r["is_private"] = r["employer_key"].eq("частное лицо")
top_emp = r.loc[~r["is_private"], "employer_key"].value_counts().head(12).to_frame("n")
top_emp["share_of_remote"] = (top_emp["n"] / len(r)).round(3)
top_emp["main_role"] = [r.loc[r["employer_key"] == e, "remote_role"].mode().iat[0] for e in top_emp.index]
print(f"Частные лица: {r['is_private'].mean():.0%} удалённых вакансий")
print(top_emp)

Частные лица: 25% удалённых вакансий
                            n  share_of_remote                                  main_role
employer_key                                                                             
т-банк                    367            0.089                    Взыскание задолженности
edler                     245            0.060  Менеджер по продажам / работе с клиентами
voxys                      77            0.019           Оператор колл-центра / поддержки
работа для каждого         29            0.007           Оператор колл-центра / поддержки
подбор персонала           14            0.003                                     Прочее
алена                      13            0.003  Менеджер по продажам / работе с клиентами
ростелеком контакт центр   12            0.003           Оператор колл-центра / поддержки
контактный центр купер     11            0.003           Оператор колл-центра / поддержки
быстрая река               11            0.003                 

**Исключаем партнёрские программы.** компания Edler (245 объявлений, 6% удалёнки) размещает десятки названий
(«Руководитель по продажам AI», «Эксперт по рекламе B2B», «Генеральный директор по развитию AI-платформы»)
с одной и той же суммой 450–600 тыс. ₽, а в тексте прямо говорится о «партнёрской программе» и «доходе от 450 000 ₽».
Это не трудовая вакансия, и одна такая компания сдвигает медиану продаж в пять раз. Правило: в тексте есть
«партнёр» или «доход от», а нижняя граница зарплаты ≥ 300 000 ₽.

**Шаблонные размещения от «разных» людей.** Например, 54 объявления «Менеджер по подбору персонала / HR-менеджер»
с одной вилкой 60–130/140 тыс. ₽ размещены под разными именами частных лиц. Число работодателей тут обманчиво,
поэтому если одинаковые основа названия и нижняя граница зарплаты встречаются у 3+ разных имён

In [ ]:
r["partner_program"] = r["text_n"].str.contains(r"партнер|доход от") & (r["salary_raw"].str.replace(r"[\s\u00a0]", "", regex=True).str.extract(r"(\d+)")[0].astype(float) >= 300_000)
print("Исключено как партнёрская программа:", int(r["partner_program"].sum()),
      r.loc[r["partner_program"], "employer_name_observed"].value_counts().head(3).to_dict())
r = r[~r["partner_program"]].copy()

STEM_STOP = r"\b(удаленно|удаленная|удаленный|удаленка|работа|на дому|подработка|от \d+ часов|без опыта|требуется|в|на|с|по)\b"
r["title_stem"] = (r["title_n"].str.replace(r"[^а-яa-z ]", " ", regex=True).str.replace(STEM_STOP, " ", regex=True)
                   .str.split().str[:3].str.join(" "))
# ключ зарплаты — нижняя граница: «60 000 — 130 000» и «60 000 — 140 000 на руки» считаем одним шаблоном
r["salary_key"] = r["salary_raw"].fillna("").str.replace(r"[\s\u00a0]", "", regex=True).str.extract(r"(\d+)")[0].fillna("нет")
tmpl = r.groupby(["title_stem", "salary_key"])["employer_name_observed"].transform("nunique")
# шаблон = одинаковые основа названия и нижняя граница зарплаты у 3+ разных имён работодателя
r["source_eff"] = np.where(tmpl >= 3, "шаблон: " + r["title_stem"] + " | " + r["salary_key"],
                           np.where(r["is_private"], "частное: " + r["employer_name_observed"].fillna(""), r["employer_key"]))
print("Источников (работодатели + шаблоны):", r["source_eff"].nunique(), "| крупнейшие шаблоны:")
print(r.loc[r["source_eff"].str.startswith("шаблон"), "source_eff"].value_counts().head(5))

Исключено как партнёрская программа: 255 {'Edler': 124, 'Edler AI': 121, 'ООО «Юрстарт»': 3}


Источников (работодатели + шаблоны): 1848 | крупнейшие шаблоны:
source_eff
шаблон: менеджер подбору персонала | 60000    54
шаблон: оператор колл центра | 35000          34
шаблон: менеджер продажам | 100000            24
шаблон: менеджер продажам | 50000             22
шаблон: менеджер продажам | 60000             16
Name: count, dtype: int64


## 5. Зарплата: формат, период и сопоставимость

Период по командному контракту: если в строке зарплаты он не указан (`unspecified`), месяцем его не считаем.
Отдельно показываем, сколько таких сумм ≥ 10 000 ₽ (кандидат «вероятно месяц») и сколько из них
подтверждается **описанием** — это первый кандидат в текстовые признаки.

In [ ]:
SALARY_RE = re.compile(
    r"(?:(от|до)\s+)?(\d[\d\s ]*)(?:\s*[—–-]\s*(\d[\d\s ]*))?\s*₽"
    r"(?:\s+(за смену|в час|за час|в неделю|за неделю|за день|в день|в месяц|за месяц|сдельно))?"
    r"(?:\s+(на руки|до вычета налогов))?", re.IGNORECASE)
UNITS = {"за смену": "shift", "в час": "hour", "за час": "hour", "в неделю": "week", "за неделю": "week",
         "за день": "day", "в день": "day", "в месяц": "month", "за месяц": "month", "сдельно": "piecework"}


def parse_salary(value):
    out = dict(sal_min=np.nan, sal_max=np.nan, sal_unit="unspecified", sal_tax="unknown", sal_shape="missing")
    if not isinstance(value, str):
        return out
    text = re.sub(r"\s+", " ", value.replace("﻿", " ")).strip()
    if not text or text.casefold() == "зарплата не указана":
        return out
    m = SALARY_RE.fullmatch(text)
    if m is None:
        out["sal_shape"] = "unrecognized"
        return out
    bound, a, b, unit, tax = m.groups()
    a = float(re.sub(r"\D", "", a))
    b = float(re.sub(r"\D", "", b)) if b else np.nan
    out["sal_unit"] = UNITS.get(unit.casefold() if unit else None, "unspecified")
    out["sal_tax"] = {"на руки": "net", "до вычета налогов": "gross"}.get(tax.casefold() if tax else None, "unknown")
    if b == b:
        out.update(sal_min=min(a, b), sal_max=max(a, b), sal_shape="range")
    elif bound and bound.casefold() == "от":
        out.update(sal_min=a, sal_shape="from")
    elif bound and bound.casefold() == "до":
        out.update(sal_max=a, sal_shape="to")
    else:
        out.update(sal_min=a, sal_max=a, sal_shape="exact")
    return out


r = pd.concat([r, r["salary_raw"].map(parse_salary).apply(pd.Series)], axis=1)
r["sal_point"] = np.where(r["sal_shape"].isin(["exact", "range"]), (r["sal_min"] + r["sal_max"]) / 2, np.nan)
r["sal_top"] = r[["sal_min", "sal_max"]].max(axis=1)

# Период из описания
P_MONTH = r"в месяц|в мес\b|/\s*мес|ежемесячн|оклад|за месяц|руб\w*\s*/\s*мес|т\.р\.? в мес|тыс\w*\s*в\s*мес"
P_HOUR = r"в час\b|за час\b|/\s*час|руб\w*\s*/\s*ч\b|почасов|за 1 час|₽/ч\b"
P_UNIT = r"за звонок|за заявк|за лид|за сделк|за урок|за заняти|за карточк|за заказ|за кандидат|за выход|за клиент|за встреч|за анкет|за продаж|за единиц"
r["desc_month"] = r["text_n"].str.contains(P_MONTH)
r["desc_hour"] = r["text_n"].str.contains(P_HOUR)
r["desc_unit"] = r["text_n"].str.contains(P_UNIT)


def period_class(row):
    if row["sal_shape"] in ("missing", "unrecognized"):
        return "нет зарплаты"
    if row["sal_unit"] != "unspecified":
        return f"явный: {row['sal_unit']}"
    if row["sal_top"] < 10_000:
        return "не указан, < 10 000 ₽ (вероятно ставка)"
    if row["desc_month"] and not (row["desc_hour"] or row["desc_unit"]):
        return "не указан, ≥ 10 000 ₽, месяц подтверждён описанием"
    if row["desc_hour"] or row["desc_unit"]:
        return "не указан, ≥ 10 000 ₽, в описании час/сдельно"
    return "не указан, ≥ 10 000 ₽, описание молчит"


r["period_class"] = r.apply(period_class, axis=1)
fmt_tbl = pd.crosstab(r["period_class"], r["sal_shape"], margins=True)
print(fmt_tbl)
fmt_tbl.to_csv(OUT / "remote_salary_formats.csv")

# Сопоставимая месячная точка для обзорных медиан:
#   strict  — явный месяц (exact/range);
#   assumed — плюс «≥ 10 000 ₽, месяц подтверждён описанием» (экспериментальное допущение для обзора, не таргет).
r["month_point_strict"] = np.where((r["sal_unit"] == "month"), r["sal_point"], np.nan)
assumed = r["period_class"].eq("не указан, ≥ 10 000 ₽, месяц подтверждён описанием")
r["month_point_assumed"] = np.where((r["sal_unit"] == "month") | assumed, r["sal_point"], np.nan)
print(f"\nАвито не пишет «в месяц» в строке зарплаты удалёнки: точек с явным месяцем {r['month_point_strict'].notna().sum()}; "
      f"с подтверждением месяца из описания: {r['month_point_assumed'].notna().sum()}")

sal_shape                                           exact  from  missing  range   to   All
period_class                                                                              
не указан, < 10 000 ₽ (вероятно ставка)                 4    29        0     12    6    51
не указан, ≥ 10 000 ₽, в описании час/сдельно           1    67        0    144   13   225
не указан, ≥ 10 000 ₽, месяц подтверждён описанием     30   340        0    705   35  1110
не указан, ≥ 10 000 ₽, описание молчит                 44   498        0    853   83  1478
нет зарплаты                                            0     0      667      0    0   667
явный: hour                                             3    32        0     16    1    52
явный: piecework                                        5    45        0     72   12   134
явный: shift                                            7    38        0     72    2   119
явный: week                                             0     3        0      9    2    14

## 6. Город или вся Россия? Сравнение внутри роли
Если удалённая зп внутри одной роли почти не зависит от города размещения, отдельные сегменты по городам не нужны:
достаточно общероссийской группы, а «Москва / Санкт-Петербург / регион» оставить признаком.

In [ ]:
r["geo3"] = np.select([r["city_resolved"].eq("Москва"), r["city_resolved"].eq("Санкт-Петербург"), r["city_resolved"].notna()],
                      ["Москва", "Санкт-Петербург", "Другие города"], default="нет города")


def med(s):
    s = s.dropna()
    return s.median() if len(s) >= 10 else np.nan


geo_cmp = (r.groupby(["remote_role", "geo3"])["month_point_assumed"]
           .agg(n="count", median=med).unstack("geo3"))
geo_cmp.columns = [f"{a}_{b}" for a, b in geo_cmp.columns]
geo_cmp["msk_vs_regions"] = geo_cmp.get("median_Москва") / geo_cmp.get("median_Другие города")
geo_cmp = geo_cmp.sort_values("n_Москва", ascending=False)
show_cols = ["n_Москва", "n_Санкт-Петербург", "n_Другие города", "median_Москва", "median_Санкт-Петербург", "median_Другие города", "msk_vs_regions"]
print(geo_cmp[[c for c in show_cols if c in geo_cmp]].round(2).to_string())

# Сколько ячеек «роль × город» проходят порог по сравнению с «роль × вся Россия»
cell_city = r.groupby(["remote_role", "city_resolved"])["month_point_assumed"].count()
cell_nat = r.groupby("remote_role")["month_point_assumed"].count()
cmp = pd.DataFrame({
    "разрез": ["роль × город", "роль × вся Россия"],
    "ячеек всего": [len(cell_city), len(cell_nat)],
    f"ячеек с ≥ {N_MIN} точками": [int((cell_city >= N_MIN).sum()), int((cell_nat >= N_MIN).sum())],
    "доля точек в таких ячейках": [round(cell_city[cell_city >= N_MIN].sum() / cell_city.sum(), 3),
                                   round(cell_nat[cell_nat >= N_MIN].sum() / cell_nat.sum(), 3)],
})
print(cmp)
geo_cmp.reset_index().to_csv(OUT / "remote_city_vs_national.csv", index=False)
cmp.to_csv(OUT / "remote_city_vs_national_summary.csv", index=False)

                                           n_Москва  n_Санкт-Петербург  n_Другие города  median_Москва  median_Санкт-Петербург  median_Другие города  msk_vs_regions
remote_role                                                                                                                                                         
Менеджер по продажам / работе с клиентами      59.0               11.0            108.0       105000.0                110000.0               90000.0            1.17
Оператор колл-центра / поддержки               36.0                4.0            128.0        60000.0                     NaN               40000.0            1.50
Прочее                                         32.0                8.0             53.0        87500.0                     NaN               70000.0            1.25
Маркетплейсы / контент                         15.0                1.0             41.0        60000.0                     NaN               82500.0            0.73
Ассистент 

## 7. Кандидаты в текстовые признаки: частота в описаниях
Простые регулярные выражения по нормализованному тексту (название + описание). Это **оценка частоты**, а не извлечение:
если признак встречается редко, извлекать его бессмысленно.

In [ ]:
TEXT_FEATURES = {
    # схема оплаты
    "оплата: оклад / фикс": r"оклад|фикс\w*|ставк[аи] \d|гарантирован",
    "оплата: % / бонус / KPI": r"%|процент|комисси|бонус|преми|kpi|кпи",
    "оплата: почасовая": P_HOUR,
    "оплата: за единицу (звонок, урок, заявку)": P_UNIT,
    "период: в месяц (из текста)": P_MONTH,
    # нагрузка и график
    "нагрузка: часы в день / неделю": r"\d+\s*(?:-|–|до)?\s*\d*\s*час\w*\s*(?:в день|в неделю|день|/день)|от \d+ час",
    "график: гибкий / свободный / подработка": r"гибк\w* график|свободн\w* график|подработ|частичн\w* занятост|неполн\w* (?:день|занятост)",
    "график: по московскому времени": r"по мск|по москв|мск\b",
    "график: ночные / выходные": r"ночн|выходн\w* дн|в выходные",
    # условия
    "оформление: официальное / ТК": r"официальн|трудов\w* договор|тк рф|по тк\b|белая (?:зарплата|зп)",
    "оформление: самозанятость / ГПХ / ИП": r"самозанят|гпх|договор подряда|\bип\b",
    "обучение: есть": r"обучени|стажировк|наставни",
    "обучение: оплачиваемое": r"оплачиваем\w* (?:обучени|стажировк)|(?:обучени|стажировк)\w* оплачива",
    "техника: свой ПК / гарнитура / интернет": r"компьютер|ноутбук|\bпк\b|гарнитур|наушник|микрофон|интернет\w* (?:от|не менее|скорост|стабильн)|стабильн\w* интернет",
    "выплаты: ежедневные / еженедельные": r"ежедневн\w* (?:выплат|оплат)|еженедельн\w* (?:выплат|оплат)|выплаты каждый день",
    # требования
    "требование: опыт продаж": r"опыт\w*\s+(?:работы\s+)?(?:в\s+)?(?:сфере\s+)?(?:продаж|звонк|телемаркет)",
    "требование: без опыта / научим": r"без опыта|научим|опыт не (?:обязателен|важен|нужен|требуется)",
    "требование: язык / предмет": r"английск|немецк|китайск|математик|физик|информатик|химии|биологи|русск\w* язык",
    "аудитория: студенты / пенсионеры / декрет": r"студент|пенсионер|декрет|мам[аы]? в декрете|инвалид",
    # специфика ролей
    "звонки: входящие": r"входящ",
    "звонки: исходящие / холодные": r"исходящ|холодн|обзвон|телемаркет",
    "продажи: B2B": r"b2b|юр\w*\s*лиц|корпоративн\w* клиент|опт\w* продаж",
    "репетитор: ЕГЭ / ОГЭ": r"егэ|огэ|экзамен",
    # риск недобросовестных объявлений
    "риск: вложения / платное обучение / пассивный доход": r"вложени|платн\w* обучени|оплат\w* обучени|пассивн\w* доход|инвестиц|без вложений|сетев\w* маркетинг",
}
feat = pd.DataFrame({k: r["text_n"].str.contains(p) for k, p in TEXT_FEATURES.items()}, index=r.index)
feat_by_role = feat.groupby(r["remote_role"]).mean().T.round(2)
feat_by_role.insert(0, "вся удалёнка", feat.mean().round(2))
print(feat_by_role.to_string())
feat_by_role.reset_index(names="признак").to_csv(OUT / "remote_text_feature_prevalence.csv", index=False)
r = pd.concat([r, feat.add_prefix("tf__")], axis=1)

remote_role                                          вся удалёнка  IT / инженер / проектировщик  Агент: недвижимость / страхование / банк  Ассистент / администратор  Бухгалтер / финансы  Взыскание задолженности  Дизайнер  Копирайтер / редактор / переводчик  Логист / диспетчер  Маркетинг / SMM / реклама  Маркетплейсы / контент  Менеджер по продажам / работе с клиентами  Надомная ручная работа  Оператор колл-центра / поддержки  Прочее  Рекрутер / HR  Репетитор / преподаватель  Юрист
оплата: оклад / фикс                                         0.30                          0.16                                      0.28                       0.31                 0.17                     0.01      0.14                                0.17                0.22                       0.26                    0.34                                       0.54                    0.04                              0.46    0.23           0.26                       0.18    0.2
оплата: % / бонус / KPI     

In [ ]:
# Поля Авито, которые уже есть в карточке (их извлекать из текста не нужно)
AVITO_FIELDS = ["Опыт работы", "Занятость", "График", "Частота выплат", "Способ оформления", "Количество рабочих часов в день",
                "Общение с клиентами", "Тип продаж", "Модель продаж", "Что получают работники", "Статус самозанятости",
                "С кем предстоит работать", "Языки программирования", "Грейд", "Канал коммуникации"]
AVITO_FIELDS = [c for c in AVITO_FIELDS if c in r.columns]
avito_fill = r.groupby("remote_role")[AVITO_FIELDS].apply(lambda g: g.notna().mean()).T.round(2)
avito_fill.insert(0, "вся удалёнка", r[AVITO_FIELDS].notna().mean().round(2))
print(avito_fill.to_string())
avito_fill.reset_index(names="поле Авито").to_csv(OUT / "remote_avito_field_fill.csv", index=False)

remote_role                      вся удалёнка  IT / инженер / проектировщик  Агент: недвижимость / страхование / банк  Ассистент / администратор  Бухгалтер / финансы  Взыскание задолженности  Дизайнер  Копирайтер / редактор / переводчик  Логист / диспетчер  Маркетинг / SMM / реклама  Маркетплейсы / контент  Менеджер по продажам / работе с клиентами  Надомная ручная работа  Оператор колл-центра / поддержки  Прочее  Рекрутер / HR  Репетитор / преподаватель  Юрист
Опыт работы                              1.00                          1.00                                      1.00                       1.00                 1.00                     1.00      1.00                                1.00                1.00                       1.00                    1.00                                       1.00                    1.00                              1.00    1.00           1.00                       1.00    1.0
Занятость                                1.00                       

## 8. Подсегменты внутри ролей
Подсегмент оправдан, если (а) разные условия труда реально по-разному оплачиваются и (б) его можно надёжно выделить

In [ ]:
def subsegment(row):
    role, t, txt = row["remote_role"], row["title_n"], row["text_n"]
    part = (row["Занятость"] == "Частичная") or bool(re.search(r"подработ|от \d+ час", t))
    if role == "Оператор колл-центра / поддержки":
        if re.search(r"продаж|исходящ|холодн|телемаркет", t + " " + txt[:600]):
            kind = "продажи / исходящие"
        elif re.search(r"поддержк|входящ|консульт|линии|линию|чат", t + " " + txt[:600]):
            kind = "поддержка / входящие"
        else:
            kind = "прочее"
        return f"{kind}; {'подработка' if part else 'полная'}"
    if role == "Менеджер по продажам / работе с клиентами":
        b2b = "B2B" if row["tf__продажи: B2B"] else "B2C / не указано"
        pay = "оклад + %" if row["tf__оплата: оклад / фикс"] else ("только %" if row["tf__оплата: % / бонус / KPI"] else "схема не указана")
        return f"{b2b}; {pay}"
    if role == "Репетитор / преподаватель":
        if re.search(r"английск|немецк|китайск|испанск|французск|язык", t):
            return "иностранные языки"
        if re.search(r"математ|физик|информатик|программир|химии|биолог", t):
            return "точные науки / IT"
        if re.search(r"куратор|наставник", t):
            return "куратор онлайн-школы"
        return "прочие предметы"
    if role == "Рекрутер / HR":
        return "массовый подбор" if re.search(r"массов", txt) else "прочий подбор"
    if role == "Маркетплейсы / контент":
        return "дизайн карточек" if re.search(r"дизайн|инфограф", t) else "менеджер / контент"
    if role == "Агент: недвижимость / страхование / банк":
        if re.search(r"недвиж|риелт|риэлт", t):
            return "недвижимость"
        if re.search(r"страх", t):
            return "страхование"
        return "банк / прочее"
    return "—"


r["subsegment"] = r.apply(subsegment, axis=1)


def top1_points(g):
    """Крупнейший источник точек: максимум из «работодатель» и «шаблон» (шаблон может разрезать одного работодателя)."""
    p = g[g["month_point_assumed"].notna()]
    if p.empty:
        return None, np.nan
    by_src = p["source_eff"].value_counts(normalize=True)
    by_emp = p.loc[~p["is_private"], "employer_key"].value_counts(normalize=True)
    if len(by_emp) and by_emp.iloc[0] > by_src.iloc[0]:
        return by_emp.index[0], by_emp.iloc[0]
    return by_src.index[0], by_src.iloc[0]


def seg_stats(g):
    pts = g["month_point_assumed"].dropna()
    pts_strict = g["month_point_strict"].dropna()
    emp = g.loc[~g["is_private"], "employer_key"].value_counts()
    salary_shown = g["sal_shape"].isin(["exact", "range", "from", "to"])
    return pd.Series({
        "n": len(g),
        "only_remote_share": (g["remote_format"] == "только удалённо").mean(),
        "employers": emp.size,
        "top1_employer": emp.index[0] if len(emp) else None,
        "top1_share": emp.iloc[0] / len(g) if len(emp) else np.nan,
        "private_share": g["is_private"].mean(),
        "salary_shown": salary_shown.mean(),
        "point_share": g["sal_shape"].isin(["exact", "range"]).mean(),
        "unit_explicit_share": (g["sal_unit"] != "unspecified").sum() / max(salary_shown.sum(), 1),
        "hour_or_unit_share": ((g["sal_unit"].isin(["hour", "piecework", "shift"])) | g["desc_hour"] | g["desc_unit"]).mean(),
        "points_month_strict": len(pts_strict),
        "points_month_assumed": len(pts),
        "points_ge10k_any": int(((g["sal_unit"] == "unspecified") & (g["sal_top"] >= 10_000) & g["sal_shape"].isin(["exact", "range"])
                                 & ~(g["desc_hour"] | g["desc_unit"])).sum()),
        "sources_with_points": g.loc[g["month_point_assumed"].notna(), "source_eff"].nunique(),
        "top1_source_points": top1_points(g)[1],
        "top1_source_name": top1_points(g)[0],
        "median_month": pts.median() if len(pts) >= 5 else np.nan,
        "q1": pts.quantile(0.25) if len(pts) >= 5 else np.nan,
        "q3": pts.quantile(0.75) if len(pts) >= 5 else np.nan,
        "msk_spb_share": g["geo3"].isin(["Москва", "Санкт-Петербург"]).mean(),
        "no_experience_share": (g["Опыт работы"] == "Без опыта").mean(),
        "part_time_share": (g["Занятость"] == "Частичная").mean(),
        "risk_flag_share": g["tf__риск: вложения / платное обучение / пассивный доход"].mean(),
    })


role_tbl = r.groupby("remote_role").apply(seg_stats).sort_values("n", ascending=False)
sub_tbl = r[r["subsegment"] != "—"].groupby(["remote_role", "subsegment"]).apply(seg_stats)
print(role_tbl[["n", "salary_shown", "points_ge10k_any", "points_month_assumed", "sources_with_points", "top1_source_points",
                "median_month", "q1", "q3", "hour_or_unit_share", "risk_flag_share"]].round(2).to_string())
print()
# (подсегменты печатаются ниже вместе с потребностью в доборе)

                                             n  salary_shown  points_ge10k_any  points_month_assumed  sources_with_points  top1_source_points  median_month        q1        q3  hour_or_unit_share  risk_flag_share
remote_role                                                                                                                                                                                                         
Менеджер по продажам / работе с клиентами  690          0.93               343                   184                  124                0.06      100000.0   65000.0  136250.0                0.09             0.05
Прочее                                     600          0.89               267                    93                   77                0.03       72500.0   50000.0  100000.0                0.15             0.06
Оператор колл-центра / поддержки           483          0.90               250                   173                   71                0.45       

## 9. Вердикт по каждому кандидату
Статусы:
* **Брать** — общероссийская группа уже проходит порог: ≥ 30 месячных точек, ≥ 10 работодателей с точками, крупнейший ≤ 30%.
* **Брать после добора** — роль понятная и однородная, но точек меньше порога; добор по явным фильтрам решит проблему.
* **Сначала признаки** — объём есть, но основная оплата почасовая/сдельная/процентная: без нагрузки или схемы оплаты из описания
  месячную сумму не сравнить.
* **Не брать** — роль неоднородная, один работодатель или высокий риск недобросовестных объявлений.

In [ ]:
FEATURES_FOR_ROLE = {
    "Оператор колл-центра / поддержки": "входящие/исходящие; часы в день; оклад vs почасовая; оплачиваемое обучение; гарнитура/ПК; подработка",
    "Менеджер по продажам / работе с клиентами": "оклад + % vs только %; B2B/B2C; холодные/тёплые (поле Авито «Тип продаж»); опыт продаж",
    "Репетитор / преподаватель": "предмет; ЕГЭ/ОГЭ; ставка за урок/час; число часов",
    "Рекрутер / HR": "массовый/IT подбор; оплата за кандидата vs оклад; опыт",
    "Маркетплейсы / контент": "площадка (Ozon/WB/Авито); дизайн карточек vs менеджер; оплата за карточку",
    "Агент: недвижимость / страхование / банк": "доля комиссии; есть ли фикс; продукт",
    "Маркетинг / SMM / реклама": "канал (поле «Канал коммуникации»); уровень; фикс vs проект",
    "Дизайнер": "направление (графический, интерьер, карточки); оплата за проект",
    "Ассистент / администратор": "часы в день; тип задач; оклад",
    "IT / инженер / проектировщик": "стек/специализация (поле «Языки программирования», «Грейд»); опыт",
    "Бухгалтер / финансы": "участок (первичка, зарплата, полный цикл); опыт; часы",
    "Взыскание задолженности": "— (одна компания, зарплата не публикуется)",
}
AVITO_FILTER_HINT = {
    "Оператор колл-центра / поддержки": "профессия «Оператор call-центра»",
    "Менеджер по продажам / работе с клиентами": "профессия «Менеджер по продажам»",
    "Репетитор / преподаватель": "профессия «Репетитор» / «Преподаватель»",
    "Рекрутер / HR": "профессия «Менеджер по персоналу» / поиск «рекрутер»",
    "Маркетплейсы / контент": "поиск «менеджер маркетплейсов»",
    "Агент: недвижимость / страхование / банк": "профессия «Агент по недвижимости»",
    "Маркетинг / SMM / реклама": "профессия «Маркетолог» / «SMM-менеджер»",
    "Дизайнер": "профессия «Дизайнер»",
    "Ассистент / администратор": "профессия «Ассистент» / «Администратор»",
    "IT / инженер / проектировщик": "профессия по специализации",
    "Бухгалтер / финансы": "профессия «Бухгалтер»",
}


def verdict(row, role):
    pts, src, top1 = row["points_month_assumed"], row["sources_with_points"], row["top1_source_points"]
    hetero = role in ("Прочее", "IT / инженер / проектировщик", "Надомная ручная работа",
                      "Копирайтер / редактор / переводчик", "Логист / диспетчер", "Юрист")
    if row["salary_shown"] < 0.3:
        return "Не брать", "зарплата почти не публикуется (одна компания, T-Банк)"
    if hetero:
        return "Не брать", "роль разнородная или слишком малая для одной группы; оставить в общем хвосте удалёнки"
    if row["risk_flag_share"] > 0.25:
        return "Не брать", f"{row['risk_flag_share']:.0%} объявлений с признаками вложений/платного обучения; доход комиссионный"
    missing = []
    if pts < N_MIN:
        missing.append(f"добрать ≥ {N_TARGET - int(pts)} вакансий с месячной зарплатой (сейчас {int(pts)})")
    if src < MIN_EMPLOYERS:
        missing.append(f"нужно ≥ {MIN_EMPLOYERS} разных источников с зарплатой (сейчас {int(src)})")
    if top1 > TOP1_MAX:
        missing.append(f"крупнейший источник «{row['top1_source_name']}» даёт {top1:.0%} точек")
    if row["hour_or_unit_share"] > 0.5:
        return "Сначала признаки", "; ".join([f"у {row['hour_or_unit_share']:.0%} оплата за час/урок — нужна ставка и нагрузка из описания"] + missing)
    if not missing:
        return "Брать", "проходит пороги в общероссийской группе"
    return "Брать после добора", "; ".join(missing)


cand = role_tbl.copy()
vv = [verdict(row, role) for role, row in cand.iterrows()]
cand["verdict"] = [v for v, _ in vv]
cand["what_is_missing"] = [m for _, m in vv]
cand["text_features_to_extract"] = [FEATURES_FOR_ROLE.get(x, "—") for x in cand.index]
cand["collection_filters"] = [
    (f"Формат работы = Удалённо; По всей России; {AVITO_FILTER_HINT[x]}; зарплата указана; "
     f"без раздвоения по городам; цель ≥ {N_TARGET} месячных точек от ≥ {MIN_EMPLOYERS} работодателей")
    if x in AVITO_FILTER_HINT else "—"
    for x in cand.index
]
order = {"Брать": 0, "Брать после добора": 1, "Сначала признаки": 2, "Не брать": 3}
cand = cand.sort_values(["verdict", "n"], key=lambda s: s.map(order) if s.name == "verdict" else -s)
show = ["n", "points_month_assumed", "sources_with_points", "top1_source_points", "median_month", "verdict", "what_is_missing"]
print(cand[show].round(2).to_string())
cand.round(3).reset_index().to_csv(OUT / "remote_candidates.csv", index=False)
sub_tbl.round(3).reset_index().to_csv(OUT / "remote_subsegments.csv", index=False)

                                             n  points_month_assumed  sources_with_points  top1_source_points  median_month             verdict                                                                                                                                                          what_is_missing
remote_role                                                                                                                                                                                                                                                                                                             
Менеджер по продажам / работе с клиентами  690                   184                  124                0.06      100000.0               Брать                                                                                                                                  проходит пороги в общероссийской группе
Ассистент / администратор                  239               

In [ ]:
# Подсегменты: где хватает точек сейчас и сколько добрать
sub = sub_tbl.copy()
sub["need_to_target"] = (N_TARGET - sub["points_month_assumed"]).clip(lower=0).astype(int)
sub["ready"] = (sub["points_month_assumed"] >= N_MIN) & (sub["sources_with_points"] >= MIN_EMPLOYERS) & (sub["top1_source_points"] <= TOP1_MAX)
print(sub[["n", "points_month_assumed", "sources_with_points", "top1_source_points", "median_month", "q1", "q3", "need_to_target", "ready"]].round(2).to_string())

                                                                                n  points_month_assumed  sources_with_points  top1_source_points  median_month        q1        q3  need_to_target  ready
remote_role                               subsegment                                                                                                                                                     
Агент: недвижимость / страхование / банк  банк / прочее                        66                     7                    7                0.17      175000.0  120000.0  262500.0              43  False
                                          недвижимость                         96                    12                   11                0.17      142500.0  100625.0  188750.0              38  False
                                          страхование                          47                     2                    2                1.00           NaN       NaN       NaN              

## 10. План адресного добора
Сколько сопоставимых точек (месячная зарплата, подтверждённая описанием) нужно добрать, чтобы роль или подсегмент прошли пороги.

In [ ]:
def need_points(row):
    pts, top1 = row["points_month_assumed"], row["top1_source_points"]
    need_n = max(N_TARGET - pts, 0)
    need_conc = 0
    if top1 == top1 and top1 > TOP1_MAX:
        need_conc = int(np.ceil(top1 * pts / TOP1_MAX - pts))   # добавить точки от других источников
    return int(max(need_n, need_conc)), int(need_conc)


SUB_FILTER_HINT = {
    ("Оператор колл-центра / поддержки", "поддержка / входящие; полная"): "поиск «оператор поддержки» / «входящие»; Занятость = Полная",
    ("Оператор колл-центра / поддержки", "поддержка / входящие; подработка"): "поиск «оператор поддержки»; Занятость = Частичная; без Voxys",
    ("Оператор колл-центра / поддержки", "продажи / исходящие; подработка"): "поиск «оператор продаж»; Занятость = Частичная",
    ("Маркетплейсы / контент", "дизайн карточек"): "поиск «дизайнер карточек» / «инфографика»",
    ("Менеджер по продажам / работе с клиентами", "B2C / не указано; только %"): "поиск «менеджер по продажам»; отбирать «только процент» — отдельный формат, не смешивать",
    ("Рекрутер / HR", "прочий подбор"): "профессия «Менеджер по персоналу»; исключить шаблон 60 000 — 130/140 000 ₽",
    ("Рекрутер / HR", "массовый подбор"): "поиск «рекрутер массовый подбор»",
    ("Репетитор / преподаватель", "точные науки / IT"): "профессия «Репетитор»; поиск «математика / физика / информатика»; приоритет — онлайн-школы с окладом",
    ("Репетитор / преподаватель", "иностранные языки"): "профессия «Репетитор»; поиск «английский»",
}
BASE_FILTERS = "Формат работы = Удалённо; география «По всей России»; зарплата указана"

plan_rows = []
for role, row in cand.iterrows():
    if row["verdict"] in ("Брать после добора", "Сначала признаки"):
        need, conc = need_points(row)
        plan_rows.append({"уровень": "роль", "remote_role": role, "subsegment": "—", "точек сейчас": int(row["points_month_assumed"]),
                          "источников": int(row["sources_with_points"]), "добрать точек": need,
                          "из них ради концентрации": conc, "фильтры": f"{BASE_FILTERS}; {AVITO_FILTER_HINT.get(role, '')}",
                          "причина": row["what_is_missing"]})
for (role, subseg), row in sub.iterrows():
    if cand.loc[role, "verdict"] == "Не брать" or row["ready"]:
        continue
    need, conc = need_points(row)
    if (role, subseg) not in SUB_FILTER_HINT:
        continue
    plan_rows.append({"уровень": "подсегмент", "remote_role": role, "subsegment": subseg, "точек сейчас": int(row["points_month_assumed"]),
                      "источников": int(row["sources_with_points"]), "добрать точек": need, "из них ради концентрации": conc,
                      "фильтры": f"{BASE_FILTERS}; {SUB_FILTER_HINT[(role, subseg)]}", "причина": "подсегмент ниже порога"})
collection_plan = pd.DataFrame(plan_rows)
print(collection_plan.drop(columns=["фильтры"]).to_string(index=False))
collection_plan.to_csv(OUT / "remote_collection_plan.csv", index=False)

   уровень                               remote_role                       subsegment  точек сейчас  источников  добрать точек  из них ради концентрации                                                                                                                                                                 причина
      роль          Оператор колл-центра / поддержки                                —           173          71             88                        88                                                                                                                              крупнейший источник «voxys» даёт 45% точек
      роль                             Рекрутер / HR                                —            59          14             88                        88                                                                                         крупнейший источник «шаблон: менеджер подбору персонала | 60000» даёт 75% точек
      роль                 Маркетинг 

In [ ]:
meta = {
    "dataset": str(DATA_PATH.relative_to(ROOT)), "sha256": DATA_SHA, "rows_total": int(len(raw)),
    "remote_rows": int(len(r)), "thresholds": {"N_MIN": N_MIN, "N_TARGET": N_TARGET, "MIN_EMPLOYERS": MIN_EMPLOYERS, "TOP1_MAX": TOP1_MAX},
    "month_point_assumed": "exact/range с явным «в месяц» ИЛИ без периода, ≥ 10 000 ₽ и с месяцем/окладом в описании без часа/сдельной оплаты",
    "script": "Analytics/remote_segments_analysis.py",
}
(OUT / "manifest.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2))
r[["source", "id", "remote_format", "remote_role", "subsegment", "employer_key", "geo3", "city_resolved",
   "sal_shape", "sal_unit", "period_class", "month_point_strict", "month_point_assumed"]].to_csv(OUT / "remote_labels.csv", index=False)
print("Сохранено в", OUT.relative_to(ROOT))

Сохранено в Analytics/remote_segment_results


## 11. Выводы

**География: город или вся Россия.** Город восстанавливается у 97% удалённых вакансий, но это город **размещения**:
511 разных городов, Москва — 28%, Санкт-Петербург — 5%, у четверти вакансий одна и та же копия размещена в нескольких городах.

для удалёнки ключ сегмента а город-признак (Москва / СПб / другие).

**Зарплата.** Авито не пишет период в строке зарплаты удалёнки: явного «в месяц» нет ни у одной вакансии.
Из 2 588 сумм ≥ 10 000 ₽ без периода месяц подтверждается описанием у 1 110, а у 225 описание говорит о почасовой или сдельной оплате.
Поэтому период из описания — **первый обязательный текстовый признак**: без него таргет для удалёнки не собрать.

**Исключить до анализа:**
* компании Edler (245 объявлений) — партнёрская программа с «доходом от 450 000 ₽», а не вакансия;
* T-Банк «Оператор раннего взыскания» (358) — один работодатель, зарплата не публикуется;
* шаблонные объявления от «разных» частных лиц

**Кандидаты:**

| Роль | Месячных точек | Источников | Медиана, ₽ | Вердикт |
|---|---|---|---|---|
| Менеджер по продажам / работе с клиентами | 184 | 124 | 100 000 | брать |
| Маркетплейсы / контент | 61 | 45 | 75 000 | брать |
| Ассистент / администратор | 52 | 48 | 45 000 | брать |
| Оператор колл-центра / поддержки | 173 | 71 | 42 500 | после добора: Voxys даёт 45% точек, нужно +88 от других |
| Рекрутер / HR | 59 | 14 | 95 000 | после добора: один шаблон даёт 75% точек, нужно +88 |
| Маркетинг / SMM | 28 | 27 | 55 600 | после добора: +22 |
| Дизайнер, бухгалтер | 10 / 8 | 7 / 8 | — | после добора: +40 / +42 |
| Репетитор / преподаватель | 11 | 11 | — | сначала признаки: у 57% оплата за час/урок |
| Агенты, взыскание, IT, логисты, надомная работа, юристы, «Прочее» | — | — | — | не брать отдельной группой |

**Текстовые признаки:**
1. Период оплаты «в месяц» / «в час» / «за единицу» — для таргета, в тексте у 35% удалёнки.
2. Схема оплаты: оклад (30%) + процент/бонус (42%) + отдельно «только процент».
3. Для репетиторов: предмет (82%), ЕГЭ/ОГЭ (55%), ставка за час или урок (33% и 16%).
4. Свой ПК / гарнитура (у операторов 76%), обучение (50%), флаг риска «вложения / платное обучение» (6%, у рекрутеров 16%).

Часы в день лучше брать из поля Авито (заполнено у 67%, у операторов — у 86%), а не из текста (11%).